### Blueheart Tools
- Notebook to go through cruise folders on blueheart and do different things, e.g.:
    - check if grids and/or Qimera projects exist
    - create lists of (missing) products like zero grids and coverage geopackages
    - generate products, if missing
    - Note that gdal sometimes is annoying to use - it may through error messages but still do the job, depending on gdal       version. Ignore the error messages if results are satifactory
- **⚡ Note that you need to be connected to Blueheart**
- **⚡ You need to have gdal installed, best via conda. Use the provided environment.yml if unsure.** 
- **⚡ Note that you need to change platform name and maybe sometimes paths to directories**
- **⚡ If you create new products, remember to update the products lists and lists of missing products again!**

In [1]:
import shutil
import geopandas as gpd
import pandas as pd
import os
import numpy as np
from pathlib import Path
import glob

In [2]:
# Set vessel name for python
platform = "MERIAN"  # "MERIAN", "METEOR", "SONNE"

### 1. Calculate zero grids
- to avoid millions of features due to colour change when vectorising
- calculates missing zero grids 
- **⚡ Genereate new zero_list & missing_zero_list afterwards (the latter should be emtpy after executing this cell successfully)**
- **⚡ Remember to change platform name variable if neccessary!**

In [ ]:
%%bash
platform="METEOR"
ifi="/Volumes/bathymetry/_blueheart/00_${platform}/${platform}_GEOMAR/${platform}_missing_zero_list.txt"
IFS=$'\n'       
set -f    
for f in $(cat < "$ifi"); do
  gdal_calc.py -A "$f" --outfile="${f%%.*}_zero.tif" --calc="(A*0)+1"
done

### 2. Polygonise zero grids
- for coverage vector/geopackage
- create list from zero grids
- polygonise zero grids to avoid millions of features due to colour change
- create list from shape files
- remove unneccessary features in shapes (those with '0')
- dissolve fields
- add field for cruise name
- **⚡ Remember to change platform name variable if neccessary!**
- **⚡ Genereate new gpkg_list & missing_gpkg_list afterwards (the latter should be emtpy)**
- **⚡ Caution: This goes through the entire missing gpkg lists and can take very long time depending of the number of files to generate.**

In [ ]:
# Copy zero grids to disk, else polygonising will take ages
dst = f"/Users/mschumacher/Docs_Data/Bathy/Processing/{platform}"
zero_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_zero_list.txt'

with open(zero_list, 'r') as z_fi:
    zero_path = z_fi.readlines()
    for z_grid in zero_path:
        z_grid = z_grid.strip("\n")
        try:
            shutil.copy(z_grid, dst)
        except Exception as e:
            print(e)
    

In [ ]:
%%bash

# change path to local zero grids and platform name accordingly

platform="METEOR"
dst="/Users/mschumacher/Docs_Data/Bathy/Processing/${platform}"
cd $dst
echo $dst
for zf in *; do
    gdal_polygonize.py "$zf" "${zf%%_zero.*}_Area.gpkg" 
done


### 4. Continue processing and adding metadata with `Add_metadata.ipynb`

#### 6. Build virtual raster tiles (vrt) and convert to geotiff to merge single raster
- **⚡ Remember to change platform name variable and paths!**
- *TODO:* Add logic to append new datasets

In [ ]:
%%bash
#platform="SONNE"
# convert single files

ifi="/Users/mschumacher/Docs_Data/Bathy/Processing/METEOR/M86-5_EM122_200m_CUBE_EPSG3395_zero.tif"
#ofi="/Volumes/bathymetry/_blueheart/00_${platform}/${platform}_GEOMAR/SO254/SO254_products/_grd/SO254_EM122_400m_CUBE_A_EPSG3395_zero.tif"
gpkg="/Users/mschumacher/Docs_Data/Bathy/Processing/METEOR/M86-5_EM122_200m_CUBE_EPSG3395_Area.gpkg"
#gdal_calc.py -A $ifi --outfile=$ofi --co="COMPRESS=DEFLATE" --co="TILED=YES" --calc="(A*0)+1"
gdal_polygonize.py $ifi $gpkg